<a href="https://colab.research.google.com/github/johnshabu720-dot/flask/blob/main/flask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import joblib

data_path = '/diabetes_prediction_dataset.csv'
if not os.path.exists(data_path):

    data_path = 'diabetes_prediction_dataset.csv'

df = pd.read_csv(data_path)
print("Dataset loaded successfully. Shape:", df.shape)
print("Columns:", df.columns.tolist())


print("\nMissing values per column:")
print(df.isnull().sum())
print("\nTarget class distribution:")
print(df['diabetes'].value_counts(normalize=True))


categorical_cols = ['gender', 'smoking_history']
numerical_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']


numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

X = df.drop('diabetes', axis=1)
y = df['diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

best_model_name = None
best_accuracy = 0
best_pipeline = None

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('model', model)])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC AUC: {auc:.4f}")
    print(classification_report(y_test, y_pred))

    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name
        best_pipeline = pipeline

print(f"\nBest Model: {best_model_name} with Accuracy: {best_accuracy:.4f}")


joblib.dump(best_pipeline, 'best_diabetes_model.pkl')
print("Best model saved to 'best_diabetes_model.pkl'")


os.makedirs('templates', exist_ok=True)


html_content = """<!DOCTYPE html>
<html>
<head>
    <title>Diabetes Prediction</title>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
</head>
<body class="bg-light">
    <div class="container py-5">
        <div class="row justify-content-center">
            <div class="col-md-6">
                <div class="card shadow">
                    <div class="card-header bg-primary text-white">
                        <h3 class="text-center">Diabetes Prediction Form</h3>
                    </div>
                    <div class="card-body">
                        {% if prediction is not none %}
                            <div class="alert alert-info text-center" role="alert">
                                <h4>Prediction Result: {{ prediction }}</h4>
                            </div>
                        {% endif %}
                        <form action="/predict" method="POST">
                            <div class="mb-3">
                                <label class="form-label">Gender</label>
                                <select name="gender" class="form-select" required>
                                    <option value="Female">Female</option>
                                    <option value="Male">Male</option>
                                    <option value="Other">Other</option>
                                </select>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">Age</label>
                                <input type="number" step="any" name="age" class="form-control" required>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">Hypertension (0 or 1)</label>
                                <select name="hypertension" class="form-select" required>
                                    <option value="0">No</option>
                                    <option value="1">Yes</option>
                                </select>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">Heart Disease (0 or 1)</label>
                                <select name="heart_disease" class="form-select" required>
                                    <option value="0">No</option>
                                    <option value="1">Yes</option>
                                </select>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">Smoking History</label>
                                <select name="smoking_history" class="form-select" required>
                                    <option value="never">Never</option>
                                    <option value="current">Current</option>
                                    <option value="former">Former</option>
                                    <option value="No Info">No Info</option>
                                </select>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">BMI</label>
                                <input type="number" step="any" name="bmi" class="form-control" required>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">HbA1c Level</label>
                                <input type="number" step="any" name="HbA1c_level" class="form-control" required>
                            </div>
                            <div class="mb-3">
                                <label class="form-label">Blood Glucose Level</label>
                                <input type="number" step="any" name="blood_glucose_level" class="form-control" required>
                            </div>
                            <button type="submit" class="btn btn-primary w-100">Predict</button>
                        </form>
                    </div>
                </div>
            </div>
        </div>
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)


flask_code = """from flask import Flask, render_template, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model = joblib.load('best_diabetes_model.pkl')

@app.route('/')
def home():
    return render_template('index.html', prediction=None)

@app.route('/predict', methods=['POST'])
def predict():
    # Get details from form
    input_data = {
        'gender': request.form.get('gender'),
        'age': float(request.form.get('age')),
        'hypertension': int(request.form.get('hypertension')),
        'heart_disease': int(request.form.get('heart_disease')),
        'smoking_history': request.form.get('smoking_history'),
        'bmi': float(request.form.get('bmi')),
        'HbA1c_level': float(request.form.get('HbA1c_level')),
        'blood_glucose_level': float(request.form.get('blood_glucose_level'))
    }

    df_input = pd.DataFrame([input_data])
    prediction = model.predict(df_input)[0]

    result = "Diabetic" if prediction == 1 else "Non-Diabetic"
    return render_template('index.html', prediction=result)

if __name__ == '__main__':
    # Run Flask on port 5000
    app.run(host='0.0.0.0', port=5000, debug=False)
"""

with open('app.py', 'w') as f:
    f.write(flask_code)

print("\nFlask app templates and 'app.py' saved successfully to current environment.")


Dataset loaded successfully. Shape: (100000, 9)
Columns: ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']

Missing values per column:
gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

Target class distribution:
diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64

=== Logistic Regression ===
Accuracy: 0.9605
ROC AUC: 0.9625
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18300
           1       0.86      0.64      0.73      1700

    accuracy                           0.96     20000
   macro avg       0.91      0.82      0.86     20000
weighted avg       0.96      0.96      0.96     20000


=== Random Forest ===
Accuracy: 0.9701
ROC AUC: 0.9604
             